## CS587 – Software Project Management (Phase 2)
## Cloud-Based Intelligent Resource Allocation System for Campus Facilities

**Domain:** Education

**Framework:** AutoGen (pyautogen)

**Model:** GPT-4.1-mini (`gpt-4.1-mini`)

**Approach:** GroupChat – Round Robin






#### Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "pyautogen==0.2.35", "openai", "tiktoken",
    "--quiet"
])
print("Libraries installed.")

Libraries installed.


#### Configuration & API Key

In [2]:
import os, json, time, re
from datetime import datetime
import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import tiktoken
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found.")

MODEL = "gpt-4.1-mini"
config_list = [{"model": MODEL, "api_key": OPENAI_API_KEY}]
llm_config = {
    "config_list": config_list,
    "temperature": 0.4,
    "max_tokens": 8192,
    "cache_seed": None,   # disables caching so every run is fresh
}
enc = tiktoken.encoding_for_model(MODEL)
print("API Key loaded.")

c:\Users\nisha\venv-langgraph\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


API Key loaded.


#### Project Domain & Scrum Configuration

In [3]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO. Sends booking/conflict notifications. Generates utilization reports.
Supports 5,000+ concurrent users and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

SCRUM_CONFIG = """
SCRUM FRAMEWORK:
- Sprint Duration: 2 weeks (10 business days)
- Total Sprints: 4 sprints
- Sprint Velocity: 40 story points per sprint
- Story Point Scale: Fibonacci (3, 5, 8)
- Definition of Done: Code reviewed, unit tested, integrated, demo-ready
- Ceremonies: Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
- Team: Scrum Master, Product Owner, Developers, QA, DevOps, UI/UX Designer
"""
print("Domain and Scrum config loaded.")

Domain and Scrum config loaded.


#### Define Scrum AI Agents

In [4]:
AGENT_SYSTEM_PROMPTS = {

"Customer_Proxy": """You are the Customer Proxy representing the university client.
Start your response with: '## CUSTOMER PROXY OUTPUT'

Provide a clear product vision statement including:
- Business problem and project goals
- Key success metrics (uptime, concurrent users, integrations)
- High-level scope from the customer perspective

Keep your response focused and concise.""",

"Product_Owner": """You are the Product Owner for a Scrum-based campus facility system.
Start your response with: '## PRODUCT OWNER OUTPUT'

Your deliverables:
1. Product Backlog with AT LEAST 24 user stories
   - Each story: ID, user story, acceptance criteria, priority, story points
   - Group into Epics: Authentication, Booking, AI/ML, Notifications, Reporting, Admin
   - Use ONLY Fibonacci story points: 3, 5, or 8 (no 13, no 21)

2. Sprint Plan for EXACTLY 4 sprints
   - Velocity = 40 story points per sprint
   - Each sprint MUST total between 36 and 40 story points
   - Every story assigned to exactly ONE sprint — NO story may be left unassigned
   - NEVER remove or skip a story — every backlog item must appear in exactly one sprint
   - Build the sprint plan in ONE pass: assign stories top to bottom, sprint by sprint
   - Do NOT rebalance or swap stories after the initial assignment
   - If a sprint reaches 36-40 SP, close it and start the next sprint
   - If the last sprint is below 36 SP, that is acceptable — do NOT attempt to fix it by moving stories
   - Show the FINAL sprint table only — do NOT show intermediate attempts or failed rebalancing steps
   - After the table, verify: count of stories in sprint plan = count in backlog

3. Definition of Done

4. Effort Calculation (at end of response):
   - Total Work = Sprint1 total + Sprint2 total + Sprint3 total + Sprint4 total
   - Do NOT recount story points from the backlog table separately
   - Verify: sum only from your final sprint plan table
   - Total Work = ___ story points
   - Velocity = 40 story points/sprint
   - Effort = 4 sprints

End with a section titled 'Effort Estimate'.""",

"Scrum_Master": """You are the Scrum Master for a Scrum-based campus facility project.
Start your response with: '## SCRUM MASTER OUTPUT'

Your deliverables:
1. Scrum Ceremonies definition:
   - Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
   - Each sprint has: 1 Planning + 10 Daily Scrums + 1 Review + 1 Retrospective = 13 ceremonies
   - 4 sprints x 13 ceremonies = 52 total ceremonies

2. Sprint Goals for ALL 4 sprints
   - Each sprint goal MUST list the exact Story IDs from the Product Owner's sprint plan
   - Format: Sprint N Goal: "...", Stories: AUTH-01, BOOK-02..., Sprint Total: XX SP

3. Velocity tracking and burndown plan

4. Effort Calculation (at end of response):
   - Total Work = 52 ceremonies
   - Productivity = 3 ceremonies/day
   - Effort = 52 / 3 = 17.33 days

End with a section titled 'Effort Estimate'.""",

"Business_Analyst": """You are the Business Analyst for a Scrum-based campus facility project.
Start your response with: '## BUSINESS ANALYST OUTPUT'

Your deliverables:
1. Detailed user stories with Given-When-Then acceptance criteria for ALL stories
2. Requirement Traceability Matrix (Story ID → Requirement → Sprint → Priority → Status)
3. Dependencies, assumptions, constraints, and compliance concerns
4. Business rules and success criteria

Effort Calculation (at end of response):
- Count EVERY story you documented with Given-When-Then criteria
- Total Work = that exact count (do NOT use a rounded or estimated number)
- Productivity = 5 requirements/day
- Effort = Total Work / 5
- Verify: your count must match the number of stories in your traceability matrix

End with a section titled 'Effort Estimate'.""",

"UI_UX_Designer": """You are the UI/UX Designer for a Scrum-based campus facility project.
Start your response with: '## UI/UX DESIGNER OUTPUT'

Your deliverables:
1. User flows for Student, Instructor, and Admin roles
2. Wireframe descriptions for AT LEAST 12 screens
   - Label each screen with a Screen ID: S1, S2, S3... up to S12 or more
   - Include: Login, Dashboard, Room Search, Booking, Confirmation, My Bookings,
     Admin Dashboard, Occupancy View, Reports, Notifications, Calendar Integration, SSO
3. Design system: typography, color palette, component library
4. WCAG 2.1 AA accessibility compliance plan
5. Sprint-wise design tasks for all 4 sprints

Effort Calculation (at end of response):
- Total Work = total number of screens designed
- Productivity = 3 screens/day
- Effort = Total Work / 3

End with a section titled 'Effort Estimate'.""",

"Developer": """You are the Developer for a Scrum-based campus facility project.
Start your response with: '## DEVELOPER OUTPUT'

Your deliverables:
1. Development tasks mapped to Product Backlog Story IDs (DEV-01, DEV-02...)
2. System architecture: Frontend (React), Backend (Node/Express), DB (MongoDB), AI/ML (Python)
3. API design: endpoints for User, Booking, Admin, AI, Integration, Notification
4. Sprint-by-sprint implementation plan following the Product Owner's sprint assignments
5. Quality gates: code review, linting, unit tests, integration tests

Effort Calculation (at end of response):
- Total Work = same story points as Product Backlog grand total
- Individual Velocity = 8 story points/developer/sprint
- Team Velocity = 40 story points/sprint (5 developers x 8)
- Effort = Total Work / 40 → round to 4 sprints

End with a section titled 'Effort Estimate'.""",

"QA_Engineer": """You are the QA Engineer for a Scrum-based campus facility project.
Start your response with: '## QA ENGINEER OUTPUT'

Your deliverables:
1. Test strategy covering functional, non-functional, performance, security, AI quality
2. AT LEAST 20 Functional Test Cases (TC-F-01, TC-F-02...)
   - Each: TC ID, Description, Related Story ID, Steps, Expected Result, Priority
   - Every story in the backlog MUST have at least one dedicated test case
   - Do NOT write "covered by other tests" — create an explicit TC for each story
   - Specifically ensure ADMIN-03 (system settings) and ADMIN-04 (audit logs) each have their own TC
3. AT LEAST 7 Non-Functional Test Cases (TC-NF-01, TC-NF-02...)
   - Cover: performance (5000+ users), uptime (99.9%), security, accessibility (WCAG 2.1 AA)
4. Full Traceability Matrix: Story ID → Test Case IDs
5. Regression plan and release quality criteria
6. Sprint-aligned testing schedule for all 4 sprints

Effort Calculation (at end of response):
- Total Work = total number of test cases
- Productivity = 3 test cases/day
- Effort = Total Work / 3

End with a section titled 'Effort Estimate'.""",

"DevOps_Engineer": """You are the DevOps Engineer for a Scrum-based campus facility project.
Start your response with: '## DEVOPS ENGINEER OUTPUT'

Your deliverables:
1. CI/CD Pipeline: GitHub Actions → Docker → Kubernetes (Dev/Staging/Production)
2. Sprint-aligned DevOps tasks table with numbered IDs:
   | Task ID | DevOps Task | Sprint | Stories Supported |
   |---------|-------------|--------|-------------------|
   | DO-01   | ...         | Sprint 1 | All |
   Include at least 16 tasks (DO-01 through DO-16 or more)
3. Monitoring: Prometheus + Grafana, ELK Stack logging
4. Security: image scanning (Trivy), secrets management (Vault)
5. Rollback strategy and operational readiness checklist

Effort Calculation (at end of response):
- Total Work = total number of DO-XX tasks
- Productivity = 8 tasks/day
- Effort = Total Work / 8

End with a section titled 'Effort Estimate'."""
}

customer_proxy = UserProxyAgent(
    name="Customer_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    default_auto_reply="",
)
product_owner = AssistantAgent(
    name="Product_Owner",
    system_message=AGENT_SYSTEM_PROMPTS["Product_Owner"],
    llm_config=llm_config,
)
scrum_master = AssistantAgent(
    name="Scrum_Master",
    system_message=AGENT_SYSTEM_PROMPTS["Scrum_Master"],
    llm_config=llm_config,
)
business_analyst = AssistantAgent(
    name="Business_Analyst",
    system_message=AGENT_SYSTEM_PROMPTS["Business_Analyst"],
    llm_config=llm_config,
)
uiux_designer = AssistantAgent(
    name="UI_UX_Designer",
    system_message=AGENT_SYSTEM_PROMPTS["UI_UX_Designer"],
    llm_config=llm_config,
)
developer = AssistantAgent(
    name="Developer",
    system_message=AGENT_SYSTEM_PROMPTS["Developer"],
    llm_config=llm_config,
)
qa_engineer = AssistantAgent(
    name="QA_Engineer",
    system_message=AGENT_SYSTEM_PROMPTS["QA_Engineer"],
    llm_config=llm_config,
)
devops_engineer = AssistantAgent(
    name="DevOps_Engineer",
    system_message=AGENT_SYSTEM_PROMPTS["DevOps_Engineer"],
    llm_config=llm_config,
)

print("Scrum AI Agents defined and created successfully.")
for name in ["Customer_Proxy", "Product_Owner", "Scrum_Master",
             "Business_Analyst", "UI_UX_Designer", "Developer",
             "QA_Engineer", "DevOps_Engineer"]:
    print(f"  - {name}")

Scrum AI Agents defined and created successfully.
  - Customer_Proxy
  - Product_Owner
  - Scrum_Master
  - Business_Analyst
  - UI_UX_Designer
  - Developer
  - QA_Engineer
  - DevOps_Engineer


#### Customer Problem Statement

In [5]:
customer_message = (
    "We need a Scrum-based project plan for our Cloud-Based Intelligent "
    "Resource Allocation System for Campus Facilities.\n\n"
    "Each agent must respond ONLY with their own role-specific Scrum artifacts.\n\n"
    "Product Owner: Create the Product Backlog with AT LEAST 24 user stories. "
    "Plan EXACTLY 4 sprints at 40 SP velocity. Each sprint must total 36-40 SP. "
    "Use only Fibonacci points: 3, 5, or 8. Show Story IDs in sprint table. "
    "CRITICAL: Every single story in the backlog MUST be assigned to a sprint. "
    "Do NOT drop or skip any story during rebalancing. "
    "Verify that the number of stories in your sprint plan equals the number in your backlog.\\n"
    "Scrum Master: Define ceremonies and sprint goals for all 4 sprints. "
    "Use exactly 10 Daily Scrums per sprint = 52 total ceremonies. "
    "List Story IDs in each sprint goal.\n"
    "Business Analyst: Write Given-When-Then acceptance criteria and traceability matrix.\n"
    "UI/UX Designer: Define wireframes for 12+ screens with IDs (S1, S2...) and design system.\n"
    "Developer: Define dev tasks (DEV-01...), architecture, APIs. "
    "Follow the Product Owner sprint assignments exactly.\n"
    "QA Engineer: Create 20+ functional (TC-F-01...) and 7+ non-functional test cases "
    "with full traceability matrix.\n"
    "DevOps Engineer: Design CI/CD pipeline with task IDs (DO-01, DO-02...) "
    "in a sprint-aligned table, at least 16 tasks.\n\n"
    "Project Scope: Students and faculty book rooms, labs, and equipment online. "
    "Administrators view real-time occupancy dashboards. AI/ML predicts demand and "
    "auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook, "
    "and SSO. Sends booking and conflict notifications. Generates utilization reports. "
    "Supports 5,000+ concurrent users and maintains 99.9% uptime."
    "CRITICAL: NOTIF-03 and NOTIF-04 must be assigned to a sprint. "
    "Do not drop any notification stories during rebalancing. "
)
print("Customer statement loaded.")

Customer statement loaded.


#### Configure GroupChat Round Robin

In [6]:
agents = [
    customer_proxy, product_owner, scrum_master,
    business_analyst, uiux_designer, developer,
    qa_engineer, devops_engineer,
]

groupchat = GroupChat(
    agents=agents,
    messages=[],
    max_round=8,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

print("GroupChat configured:")
print(f"  Speaker selection : round_robin")
print(f"  Max rounds        : {groupchat.max_round}")
print(f"  Agents            : {[a.name for a in agents]}")

GroupChat configured:
  Speaker selection : round_robin
  Max rounds        : 8
  Agents            : ['Customer_Proxy', 'Product_Owner', 'Scrum_Master', 'Business_Analyst', 'UI_UX_Designer', 'Developer', 'QA_Engineer', 'DevOps_Engineer']


#### Run Experiment

In [7]:
import shutil

# Reset all agents completely
for agent in [customer_proxy, product_owner, scrum_master,
              business_analyst, uiux_designer, developer,
              qa_engineer, devops_engineer]:
    agent.reset()

# Clear AutoGen cache
cache_dir = ".cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Cache cleared!")
else:
    print("No cache to clear.")

# Rebuild GroupChat and Manager fresh
groupchat = GroupChat(
    agents=agents,
    messages=[],
    max_round=8,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

print("Starting fresh Scrum GroupChat experiment...")
print("=" * 60)

start_time = time.time()

customer_proxy.initiate_chat(
    manager,
    message=customer_message,
)

total_elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"  Scrum GroupChat completed in {total_elapsed:.1f}s")
print("=" * 60)

No cache to clear.
Starting fresh Scrum GroupChat experiment...
Customer_Proxy (to chat_manager):

We need a Scrum-based project plan for our Cloud-Based Intelligent Resource Allocation System for Campus Facilities.

Each agent must respond ONLY with their own role-specific Scrum artifacts.

Product Owner: Create the Product Backlog with AT LEAST 24 user stories. Plan EXACTLY 4 sprints at 40 SP velocity. Each sprint must total 36-40 SP. Use only Fibonacci points: 3, 5, or 8. Show Story IDs in sprint table. CRITICAL: Every single story in the backlog MUST be assigned to a sprint. Do NOT drop or skip any story during rebalancing. Verify that the number of stories in your sprint plan equals the number in your backlog.\nScrum Master: Define ceremonies and sprint goals for all 4 sprints. Use exactly 10 Daily Scrums per sprint = 52 total ceremonies. List Story IDs in each sprint goal.
Business Analyst: Write Given-When-Then acceptance criteria and traceability matrix.
UI/UX Designer: Define 

#### Save Results

In [8]:
output_dir = "Experiment_Results_1"
os.makedirs(output_dir, exist_ok=True)

agent_names = [
    "Product_Owner", "Scrum_Master", "Business_Analyst",
    "UI_UX_Designer", "Developer", "QA_Engineer", "DevOps_Engineer",
]

agent_system_prompts = {
    a.name: a.system_message
    for a in agents
    if hasattr(a, "system_message") and a.system_message
}

all_results = []
for msg in groupchat.messages:
    name = msg.get("name", "")
    content = msg.get("content", "")
    if name in agent_names and content:
        output_tokens = len(enc.encode(content))
        sys_prompt = agent_system_prompts.get(name, "")
        input_tokens = len(enc.encode(sys_prompt)) + len(enc.encode(customer_message))
        total_tokens_agent = input_tokens + output_tokens
        all_results.append({
            "agent": name,
            "response": content,
            "tokens": total_tokens_agent,
            "model": MODEL,
        })

total_tokens = sum(r["tokens"] for r in all_results)

for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']} | **Tokens Used:** {result['tokens']:,}\n\n")
        f.write("---\n\n")
        f.write(result["response"])
    print(f"Saved: {filepath}")

section_names = {
    "Product_Owner":    "SECTION 1 – PRODUCT OWNER (PRODUCT BACKLOG)",
    "Scrum_Master":     "SECTION 2 – SCRUM MASTER (CEREMONIES & SPRINT GOALS)",
    "Business_Analyst": "SECTION 3 – BUSINESS ANALYST (USER STORIES & TRACEABILITY)",
    "UI_UX_Designer":   "SECTION 4 – UI/UX DESIGNER (WIREFRAMES & DESIGN SYSTEM)",
    "Developer":        "SECTION 5 – DEVELOPER (SPRINT TASKS & ARCHITECTURE)",
    "QA_Engineer":      "SECTION 6 – QA ENGINEER (TEST PLAN)",
    "DevOps_Engineer":  "SECTION 7 – DEVOPS ENGINEER (CI/CD & INFRASTRUCTURE)",
}

combined_path = os.path.join(output_dir, "Phase2_Scrum_Complete_Report.md")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 2 – Scrum Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write("**Framework:** AutoGen (pyautogen) – GroupChat Round Robin\n")
    f.write("**Methodology:** Scrum (Agile Iterative Development) – 4 Sprints\n\n---\n\n")
    f.write("## SECTION 0 – CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message + "\n\n---\n\n")
    for result in all_results:
        section = section_names.get(result["agent"], result["agent"])
        f.write(f"## {section}\n\n")
        f.write(f"**Model:** {result['model']} | **Tokens:** {result['tokens']:,}\n\n")
        f.write(result["response"] + "\n\n---\n\n")

metadata = {
    "experiment": {
        "title": "Phase 2 – Scrum Project Planning (Exp 1)",
        "project": "Cloud-Based Intelligent Resource Allocation System",
        "model": MODEL,
        "framework": f"AutoGen (pyautogen v{autogen.__version__}) – GroupChat Round Robin",
        "speaker_selection": "round_robin",
        "sprints": 4,
        "date": datetime.now().isoformat(),
        "total_tokens": total_tokens,
        "total_time_seconds": round(total_elapsed, 2),
    },
    "agents": [{"name": r["agent"], "tokens": r["tokens"], "model": r["model"]} for r in all_results],
    "groupchat": {
        "total_messages": len(groupchat.messages),
        "max_round": groupchat.max_round,
    },
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"\nCombined report saved: {combined_path}")
print(f"Metadata saved:        {metadata_path}")
print(f"Total tokens used:     {total_tokens:,}")

Saved: Experiment_Results_1\Product_Owner_output.md
Saved: Experiment_Results_1\Scrum_Master_output.md
Saved: Experiment_Results_1\Business_Analyst_output.md
Saved: Experiment_Results_1\UI_UX_Designer_output.md
Saved: Experiment_Results_1\Developer_output.md
Saved: Experiment_Results_1\QA_Engineer_output.md
Saved: Experiment_Results_1\DevOps_Engineer_output.md

Combined report saved: Experiment_Results_1\Phase2_Scrum_Complete_Report.md
Metadata saved:        Experiment_Results_1\experiment_metadata.json
Total tokens used:     18,406


#### Work Effort, Duration & Effort Analysis

In [9]:
HOURS_PER_PT = 4
WD = 8
dur_min = total_elapsed / 60
avg_per_agent = total_elapsed / len(all_results) if all_results else 0
NUM_SPRINTS = 4
SPRINT_WEEKS = 2
proj_weeks = NUM_SPRINTS * SPRINT_WEEKS
proj_budget_hrs = proj_weeks * 5 * WD

po = next((r['response'] for r in all_results if r['agent'] == 'Product_Owner'), '')
total_sp_match = re.search(r'Total Work\s*=\s*(\d+)\s*story points', po, re.IGNORECASE)
total_sp = int(total_sp_match.group(1)) if total_sp_match else 0
avg_vel = total_sp / NUM_SPRINTS if total_sp else 0

print('=' * 62)
print('  WORK EFFORT, DURATION & EFFORT ANALYSIS')
print('=' * 62)
print(f'\n📅 DURATION')
print(f'  Sprints Planned         : {NUM_SPRINTS} x {SPRINT_WEEKS}-week sprints')
print(f'  Total Project Duration  : {proj_weeks} weeks ({proj_weeks*5} working days / {proj_weeks*7} calendar days)')
print(f'  Capacity Budget         : {proj_budget_hrs} work-hours')
print(f'  AI Experiment (wall-clock): {total_elapsed:.1f}s  ({dur_min:.1f} min)')
print(f'  Avg Time per Agent      : {avg_per_agent:.1f}s')
print(f'\n🎯 WORK EFFORT  (Story Points)')
print(f'  Total Story Points      : {total_sp} pts')
print(f'  Team Velocity           : 40 story points/sprint')
print(f'  Sprints Needed          : {NUM_SPRINTS} sprints')
print(f'  Avg Velocity            : {avg_vel:.1f} pts/sprint')
print(f'\n⏱  EFFORT ESTIMATION  (@ {HOURS_PER_PT}h per story point)')
th = total_sp * HOURS_PER_PT
print(f'  Total Estimated Effort  : {th} hrs  ({th/WD:.1f} days)  ({th/WD/5:.1f} weeks)')
print(f'  Capacity Utilisation    : {th/proj_budget_hrs*100:.1f}%   ({th}/{proj_budget_hrs} hrs)')
print(f'\n🤖 AGENT TOKEN EFFORT')
print(f"  {'Agent':<22} {'Tokens':>8} {'%':>6} {'Chars':>8}")
print(f"  {'-'*50}")
for r in all_results:
    pct = r['tokens'] / total_tokens * 100 if total_tokens else 0
    print(f"  {r['agent']:<22} {r['tokens']:>8,}  {pct:>5.1f}%  {len(r['response']):>8,}")
print(f"  {'-'*50}")
print(f"  {'TOTAL':<22} {total_tokens:>8,}  100.0%")
print(f"  Throughput : {total_tokens/total_elapsed:.1f} tok/s")
print(f"  Cost proxy : ${total_tokens/1e6*0.40:.4f} USD   (GPT-4.1-mini @ $0.40/1M tokens)")
print('=' * 62)

  WORK EFFORT, DURATION & EFFORT ANALYSIS

📅 DURATION
  Sprints Planned         : 4 x 2-week sprints
  Total Project Duration  : 8 weeks (40 working days / 56 calendar days)
  Capacity Budget         : 320 work-hours
  AI Experiment (wall-clock): 185.4s  (3.1 min)
  Avg Time per Agent      : 26.5s

🎯 WORK EFFORT  (Story Points)
  Total Story Points      : 138 pts
  Team Velocity           : 40 story points/sprint
  Sprints Needed          : 4 sprints
  Avg Velocity            : 34.5 pts/sprint

⏱  EFFORT ESTIMATION  (@ 4h per story point)
  Total Estimated Effort  : 552 hrs  (69.0 days)  (13.8 weeks)
  Capacity Utilisation    : 172.5%   (552/320 hrs)

🤖 AGENT TOKEN EFFORT
  Agent                    Tokens      %    Chars
  --------------------------------------------------
  Product_Owner             3,517   19.1%    11,585
  Scrum_Master              1,225    6.7%     2,328
  Business_Analyst          3,145   17.1%    11,994
  UI_UX_Designer            1,777    9.7%     5,451
  Develo

#### Experiment Summary

In [10]:
print('=' * 62)
print('  SCRUM EXPERIMENT 1 COMPLETE')
print('=' * 62)
print()
print(f'  Model         : {MODEL}')
print(f'  Framework     : AutoGen (pyautogen v{autogen.__version__})')
print(f'  Method        : Scrum (Agile) – 4 Sprints')
print(f'  Interaction   : GroupChat Round Robin')
print(f'  Total Agents  : {len(all_results)}')
print(f'  Total Messages: {len(groupchat.messages)}')
print(f'  Total Tokens  : {total_tokens:,}')
print(f'  Total Time    : {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Tokens/sec    : {total_tokens/total_elapsed:.1f}')
print()
po_response = next((r['response'] for r in all_results if r['agent'] == 'Product_Owner'), '')
total_sp_match = re.search(r'Total Work\s*=\s*(\d+)\s*story points', po_response, re.IGNORECASE)
total_sp = int(total_sp_match.group(1)) if total_sp_match else 0
num_sprints = 4
sprint_weeks = 2
total_weeks = num_sprints * sprint_weeks
print(f'  -- Work Effort --')
print(f'  Total Story Points : {total_sp} pts')
print(f'  Team Velocity      : 40 story points/sprint')
print(f'  Sprints Needed     : {num_sprints} sprints')
print(f'  Total Effort       : {total_sp * 4} hrs  ({total_sp * 4 / 8:.1f} days)')
print(f'  Avg Velocity       : {total_sp / num_sprints:.1f} pts/sprint')
print()
print(f'  -- Duration --')
print(f'  Scrum Plan   : {num_sprints} sprints x {sprint_weeks} weeks = {total_weeks} weeks')
print(f'  Working Days : {total_weeks*5} working days / {total_weeks*7} calendar days')
print(f'  AI Experiment: {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Avg/Agent    : {total_elapsed/len(all_results):.1f}s')
print()
print(f'  -- Agent Breakdown --')
for r in all_results:
    print(f"    {r['agent']:<22s} | Tokens: {r['tokens']:>7,} | Chars: {len(r['response']):>6,}")
print()
print(f'  Output Dir: {output_dir}/')
print('=' * 62)

  SCRUM EXPERIMENT 1 COMPLETE

  Model         : gpt-4.1-mini
  Framework     : AutoGen (pyautogen v0.2.35)
  Method        : Scrum (Agile) – 4 Sprints
  Interaction   : GroupChat Round Robin
  Total Agents  : 7
  Total Messages: 8
  Total Tokens  : 18,406
  Total Time    : 185.4s  (3.1 min)
  Tokens/sec    : 99.3

  -- Work Effort --
  Total Story Points : 138 pts
  Team Velocity      : 40 story points/sprint
  Sprints Needed     : 4 sprints
  Total Effort       : 552 hrs  (69.0 days)
  Avg Velocity       : 34.5 pts/sprint

  -- Duration --
  Scrum Plan   : 4 sprints x 2 weeks = 8 weeks
  Working Days : 40 working days / 56 calendar days
  AI Experiment: 185.4s  (3.1 min)
  Avg/Agent    : 26.5s

  -- Agent Breakdown --
    Product_Owner          | Tokens:   3,517 | Chars: 11,585
    Scrum_Master           | Tokens:   1,225 | Chars:  2,328
    Business_Analyst       | Tokens:   3,145 | Chars: 11,994
    UI_UX_Designer         | Tokens:   1,777 | Chars:  5,451
    Developer             